# 01 Targets set in the calls

The basis of the comparison are the 39 quantified targets that UBS management set in the earnings calls from 2023-Q2 to 2024-Q4, after the acquisition of Credit Suisse. 31 were found by fixed rules (a quantity, a future horizon and a forward-looking word in one sentence) and 8 by the FinBERT-FLS model among the sentences that the rules missed; each was read and judged genuine. This notebook lists them and looks up every source sentence in the export of 2026-09-30.

In [1]:
# project setup - run first, do not edit except NAME
NAME = "call_only_targets"

import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO = "boe-financial-analysis"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if not pathlib.Path(f"/content/{REPO}").exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/maybebool/{REPO}.git",
                        f"/content/{REPO}"], check=True)
    ROOT = pathlib.Path(f"/content/{REPO}")
    DATA = pathlib.Path("/content/drive/MyDrive/boe-data")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=False)
    subprocess.run(["git", "-C", str(ROOT), "merge", "-q", "origin/main", "-m", "sync"],
                   check=False)
else:
    ROOT = pathlib.Path.cwd()
    while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA = ROOT / "data"

sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)

from notebooks.env_cell import verify, check_python
check_python()
try:
    verify(ROOT)
except RuntimeError as e:
    print(e)
    print("\n Runtime -> Restart session, then run this cell again.")
    raise

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import loading

print(f"ok  python {sys.version.split()[0]}  pandas {pd.__version__}")

ok  python 3.13.15  pandas 2.2.3


In [2]:
# folders and helpers shared by the notebooks of this section
import json, re
NB_DIR = ROOT / "notebooks" / "05_call_only_targets"
sys.path.insert(0, str(NB_DIR))
import report_checks as rc

IN = DATA / "call_only_targets"        # delivered result files
REPORTS = DATA / "reports"             # report files, optional (see 00_get_reports)
EXPORTS = NB_DIR / "exports"
EXPORTS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 160, "display.max_rows", 60)

manifest = pd.read_csv(IN / "reports_manifest.csv", keep_default_na=False)
status = rc.report_status(REPORTS, manifest)
print(rc.mode_message(status))

REPORTS PRESENT (29 of 29): every quote is verified against the text of its page.


## The 39 targets

In [3]:
targets = pd.read_csv(IN / "targets_39.csv", keep_default_na=False)
targets.groupby("found_by").size().rename("targets").to_frame().T

found_by,FinBERT-FLS,rules
targets,8,31


In [4]:
targets[["unit", "call", "found_by", "figure_call", "call_sentence"]]

,unit,call,found_by,figure_call,call_sentence
0,T16,2023-Q3,rules,NCL operational risk RWA around 14bn by end-2026,"On the basis of natural roll-off in both contexts, we expect operational risk RWA in NCL to decrease to around 14 billion by the end of 2026."
1,T25,2023-Q4,rules,NCL capital release over 6bn by end-2026,"Obviously, the most prominent example is the Non-core and Legacy portfolio, where we expect our wind-down efforts to result in a capital release of over 6 b..."
2,T27,2023-Q4,rules,NCL credit and market risk RWA substantially below 40bn by end-2024,"By the end of 2024, we expect combined risk – credit and market risk risk-weighted assets to be substantially below 40 billion."
3,T32,2023-Q4,rules,going-concern requirement +180bp to 16.7% between 2026 and 2030,Between 2026 and 2030 our going-concern capital requirement is expected to increase by around 180 basis points to 16.7% as the effects of the currently larg...
4,T33,2023-Q4,rules,AT1 issuance up to 2bn in 2024,"In this respect, following last year’s successful raises, we expect to issue up to 2 billion in AT1 in 2024."
5,T68,2024-Q4,rules,Group HoldCo around 90bn by end-2025,"Looking ahead, we’re targeting to bring down Group HoldCo to around 90 billion by the end of 2025 while still retaining resilient buffers over regulatory mi..."
6,T10,2023-Q2,rules,CET1 ratio around 14% over the medium term,We expect to operate at around 14% CET1 capital ratio over the medium term.
7,T12,2023-Q2,rules,over 3bn exit-rate saves by end-2023,"So, Andrew, in terms of – I’ll take the second point first – in terms of the story on the underlying profitability, yeah, I mean just be very clear that the..."
8,T13,2023-Q2,rules,over 3bn exit-rate saves by end-2023,But as I would – the way I would think about it is there is work that's ongoing and we expect that the greater than 3 billion number is something that we'll...
9,T9,2023-Q2,rules,cost/income ratio below 70% in 2026,"All told, we aim to bring the Group’s underlying cost/income ratio exit rate below 70% in 2026."


## Source sentences in the export of 2026-09-30

In [5]:
EXPORT = "2026-09-30"
sentences = loading.load(DATA, "all_sentences.csv", EXPORT)
match = pd.read_csv(IN / "export_match.csv", keep_default_na=False)

QUOTES = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"'})
BRACKET = re.compile(r"\[[^\]]*\]")
def norm(text):
    """Editorial insertions in square brackets removed, straight quotation marks, single spaces."""
    return re.sub(r"\s+", " ", BRACKET.sub(" ", str(text)).translate(QUOTES)).strip()

ubs = sentences[(sentences.bank == "UBS") & (sentences.call_type == "earnings")]
key = ["quarter", "position_in_call", "sentence_number"]
look = match.assign(quarter=match.call.str.split().str[1],
                    position_in_call=match.position_in_call_new.astype(int),
                    sentence_number=match.sentence_number_new.astype(int))
found = look.merge(ubs[key + ["section", "speaker_name", "sentence"]], on=key, how="left", validate="one_to_one")
# a statement can span two export sentences where the splitter cut after "ex."; the first one must then open it
found["same_wording"] = [norm(t) == norm(s) or (norm(s).endswith(" ex.") and norm(t).startswith(norm(s)))
                         for t, s in zip(found.text_old, found.sentence.fillna(""))]
print("source sentences found in the export:", int(found.sentence.notna().sum()), "of", len(found))
print("with the same wording:", int(found.same_wording.sum()), "| with the same speaker:",
      int((found.speaker_name == found.speaker_old).sum()))

loaded all_sentences.csv  19092 rows
source sentences found in the export: 39 of 39
with the same wording: 39 | with the same speaker: 39


All 39 source sentences are present in the export of 2026-09-30 with the same wording and speaker. The targets were extracted on an earlier export; in the seven UBS calls of the window the two exports are identical in text (`export_check_summary.json`).

In [6]:
summary = json.loads((IN / "export_check_summary.json").read_text())
pd.DataFrame({k: summary[k] for k in ("sentences", "rule_candidates", "set_s")}).T[["old", "new", "same"]]

,old,new,same
sentences,4175,4175,4175
rule_candidates,105.0,105.0,105.0
set_s,68.0,68.0,68.0


## Export

In [7]:
out = targets.merge(found[["unit", "section", "speaker_name", "position_in_call", "sentence_number"]], on="unit")
out = out.rename(columns={"speaker_name": "speaker"})
out.to_csv(EXPORTS / "targets_39.csv", index=False)
out.groupby(["call", "section"]).size().unstack(fill_value=0)

section,prepared,qa
call,,
2023-Q2,2,2
2023-Q3,4,0
2023-Q4,19,0
2024-Q1,2,0
2024-Q2,2,0
2024-Q3,2,0
2024-Q4,6,0


In [8]:
# checks against the primary values
assert len(targets) == 39 and targets.unit.is_unique
assert targets.found_by.value_counts().to_dict() == {"rules": 31, "FinBERT-FLS": 8}
assert found.sentence.notna().all() and found.same_wording.all()
assert (found.speaker_name == found.speaker_old).all()
assert summary["sentences"]["old"] == summary["sentences"]["new"] == summary["sentences"]["same"] == 4175
assert summary["only_new_for_reading"] == 0 and summary["only_old"] == 0
print("all checks passed")

all checks passed
